# 爬 Investing.com 資料

In [3]:
# %pip install selenium
# %pip install webdriver_manager

In [39]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
import time



# Set up the browser (proper WebDriver instantiation)
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

# Open the Dow Jones historical data page
url = "https://www.investing.com/indices/us-30-historical-data"
driver.get(url)

# Optional: wait for 5 seconds before proceeding
driver.implicitly_wait(5)



print("find date")
# Locate the date input fields (these XPaths or element IDs may vary)
# Click on the date input to open the calendar and set start date
start_date = driver.find_element(By.XPATH, '/html/body/div[2]/div[2]/div[1]/div[2]/div[1]/div[2]/div[2]/div[2]/div[2]/div[3]/div[1]/div[1]/input')
start_date.click()
start_date.clear()
start_date.send_keys("01/01/2022")
start_date.send_keys(Keys.RETURN)

# Set end date
end_date = driver.find_element(By.XPATH, '/html/body/div[2]/div[2]/div[1]/div[2]/div[1]/div[2]/div[2]/div[2]/div[3]/div[1]/div[2]/input')
end_date.click()
end_date.clear()
end_date.send_keys("01/01/2023")
end_date.send_keys(Keys.RETURN)

# Click apply (assuming the 'Apply' button has a recognizable selector)
apply_button = driver.find_element(By.XPATH, '//*[@id="applyBtn"]')
apply_button.click()

# Wait for the data to load
time.sleep(10)

# Optionally: Scrape the table data
table = driver.find_element(By.XPATH, '//*[@id="curr_table"]')
rows = table.find_elements(By.TAG_NAME, "tr")

# Process rows and scrape data
for row in rows:
    cols = row.find_elements(By.TAG_NAME, "td")
    print([col.text for col in cols])

# Close the browser
driver.quit()

find date


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=129.0.6668.90)
Stacktrace:
0   chromedriver                        0x00000001011ac248 cxxbridge1$str$ptr + 1907280
1   chromedriver                        0x00000001011a4730 cxxbridge1$str$ptr + 1875768
2   chromedriver                        0x0000000100db8260 cxxbridge1$string$len + 89488
3   chromedriver                        0x0000000100d93728 core::str::slice_error_fail::hbaf5d05fe3921cd2 + 3776
4   chromedriver                        0x0000000100e22e8c cxxbridge1$string$len + 526780
5   chromedriver                        0x0000000100e36218 cxxbridge1$string$len + 605512
6   chromedriver                        0x0000000100df112c cxxbridge1$string$len + 322652
7   chromedriver                        0x0000000100df1d7c cxxbridge1$string$len + 325804
8   chromedriver                        0x00000001011744d8 cxxbridge1$str$ptr + 1678560
9   chr

# 畫技術分析


In [2]:
import pandas as pd
import numpy as np
import json
import requests
import time
import os


def cal_tech(id):

    path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw', 'stock_price')
    # folder_list = ['2330', '2454', '2317', '2382', '2308', '3711']


    stock_tech = pd.read_csv(os.path.join(path , id + 'twse.csv'))

    # # 將stock_tech的值都換成float
    # stock_tech['Open'] = stock_tech['Open'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['Close'] = stock_tech['Close'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['High'] = stock_tech['High'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['Low'] = stock_tech['Low'].map(lambda x: float(x.replace(',', '')))


    # 計算移動均線 (SMA)
    stock_tech['MA5'] = stock_tech['Close'].rolling(window=5).mean()
    stock_tech['MA10'] = stock_tech['Close'].rolling(window=10).mean()
    stock_tech['MA20'] = stock_tech['Close'].rolling(window=20).mean()

    # 計算指數移動平均線 (EMA)
    stock_tech['EMA5'] = stock_tech['Close'].ewm(span=5, adjust=False).mean()
    stock_tech['EMA12'] = stock_tech['Close'].ewm(span=12, adjust=False).mean()
    stock_tech['EMA26'] = stock_tech['Close'].ewm(span=26, adjust=False).mean()

    # 計算MACD和DIF
    stock_tech['DIF'] = stock_tech['EMA12'] - stock_tech['EMA26']
    stock_tech['MACD'] = stock_tech['DIF'].ewm(span=9, adjust=False).mean()


    # 計算RSI
    def calculate_rsi(series, period=14):
        delta = series.diff(1)
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi

    stock_tech['RSI'] = calculate_rsi(stock_tech['Close'])

    # 計算KDJ
    low_list = stock_tech['Low'].rolling(9, min_periods=9).min()
    low_list.fillna(value=stock_tech['Low'].expanding().min(), inplace=True)
    high_list = stock_tech['High'].rolling(9, min_periods=9).max()
    high_list.fillna(value=stock_tech['High'].expanding().max(), inplace=True)
    rsv = (stock_tech['Close'] - low_list) / (high_list - low_list) * 100


    # 初始值設定
    stock_tech['K'] = 50
    stock_tech['D'] = 50

    for i in range(1, len(stock_tech)):
        stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
        stock_tech.loc[stock_tech.index[i], 'D'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']


    # 顯示DataFrame
    stock_tech.to_csv(os.path.join(path , id + 'tech.csv'), encoding='utf-8', index=False)

In [3]:
path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw', 'stock_price')
folder_list = ['6505']

for id in folder_list:
    try:
        stock_tech = fetch_data_twse(id, 2022, 2024) # 這裡可以自行設定要抓取的時間範圍
        stock_tech.to_csv(os.path.join(path , id + 'twse.csv')) # 將資料存成 csv 檔
        print('save ' + id + 'twse.csv')
        cal_tech(id)
        print('save ' + id + 'tech.csv')
    except Exception as e:
        print("Error: " + id + " " + str(e))
        pass

Fetching data for 6505 from 2022 to 2024...
Data not found for 202410
save 6505twse.csv
save 6505tech.csv
Fetching data for 2357 from 2022 to 2024...


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_34443/2247075068.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '51.11111111111104' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_34443/2247075068.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '50.37037037037034' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'D'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']


Data not found for 202410
save 2357twse.csv
save 2357tech.csv
Fetching data for 1216 from 2022 to 2024...


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_34443/2247075068.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '59.52380952380952' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_34443/2247075068.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '53.17460317460316' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'D'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']


Data not found for 202410
save 1216twse.csv
save 1216tech.csv
Fetching data for 3045 from 2022 to 2024...


...

/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_34443/2247075068.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '45.83333333333326' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_34443/2247075068.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '48.61111111111108' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'D'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']


Data not found for 202410
save 2207twse.csv
save 2207tech.csv
Fetching data for 2880 from 2022 to 2024...


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_34443/2247075068.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '52.38095238095237' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_34443/2247075068.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '50.793650793650784' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'D'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']


Data not found for 202410
save 2880twse.csv
save 2880tech.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_34443/2247075068.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '44.44444444444457' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_34443/2247075068.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '48.14814814814818' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'D'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']
